# SafeDrive AI ML POC: Distraction Detection
Demonstrate face landmark detection using MediaPipe FaceMesh and outline workflow for distraction classification with MobileNetV2.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install mediapipe opencv-python tensorflow

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from matplotlib import pyplot as plt

# Load sample image
img = cv2.imread('sample_face.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Run MediaPipe FaceMesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True)
results = face_mesh.process(img_rgb)

# Draw landmarks
if results.multi_face_landmarks:
    for face_landmarks in results.multi_face_landmarks:
        for lm in face_landmarks.landmark:
            x = int(lm.x * img.shape[1])
            y = int(lm.y * img.shape[0])
            cv2.circle(img, (x, y), 1, (0,255,0), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('FaceMesh Landmarks')
plt.axis('off')
plt.show()

## Part 2: Drowsiness Detection (PERCLOS/EAR)
Calculate Eye Aspect Ratio (EAR) to detect drowsiness when eyes are closed for extended periods.

In [ ]:
def calculate_ear(eye_landmarks):
    """
    Calculate Eye Aspect Ratio (EAR) for drowsiness detection.
    EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)
    where p1-p6 are eye landmark points.
    """
    # Vertical eye distances
    A = np.linalg.norm(np.array(eye_landmarks[1]) - np.array(eye_landmarks[5]))
    B = np.linalg.norm(np.array(eye_landmarks[2]) - np.array(eye_landmarks[4]))
    # Horizontal eye distance
    C = np.linalg.norm(np.array(eye_landmarks[0]) - np.array(eye_landmarks[3]))
    
    ear = (A + B) / (2.0 * C)
    return ear

def extract_eye_landmarks(face_landmarks, eye_indices):
    """Extract eye landmark coordinates from MediaPipe face mesh."""
    return [(face_landmarks.landmark[i].x, face_landmarks.landmark[i].y) for i in eye_indices]

# MediaPipe eye landmark indices (simplified - use actual indices from MediaPipe)
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# Simulate EAR calculation (requires actual face landmarks from MediaPipe)
if results.multi_face_landmarks:
    for face_landmarks in results.multi_face_landmarks:
        left_eye_coords = extract_eye_landmarks(face_landmarks, LEFT_EYE)
        right_eye_coords = extract_eye_landmarks(face_landmarks, RIGHT_EYE)
        
        left_ear = calculate_ear(left_eye_coords)
        right_ear = calculate_ear(right_eye_coords)
        avg_ear = (left_ear + right_ear) / 2.0
        
        print(f"Left EAR: {left_ear:.3f}, Right EAR: {right_ear:.3f}, Avg EAR: {avg_ear:.3f}")
        
        # Drowsiness threshold: EAR < 0.25 for 3+ seconds indicates drowsiness
        EAR_THRESHOLD = 0.25
        if avg_ear < EAR_THRESHOLD:
            print("⚠️ DROWSINESS DETECTED - Eyes closing!")
        else:
            print("✓ Alert and awake")

## Part 3: Crash Detection (Accelerometer-Based)
Detect crashes using multi-stage validation of accelerometer data: peak detection → validation → classification.

In [ ]:
import time

def detect_crash(accel_data, threshold=2.5):
    """
    Multi-stage crash detection algorithm:
    1. Peak Detection: Look for sudden acceleration spikes
    2. Validation: Check sustained high acceleration
    3. Classification: Determine crash severity
    
    Args:
        accel_data: Array of (x, y, z) accelerometer readings in G-force
        threshold: G-force threshold for crash detection (default 2.5G)
    
    Returns:
        crash_detected (bool), severity (str)
    """
    crash_detected = False
    severity = "none"
    
    for i, (x, y, z) in enumerate(accel_data):
        # Calculate total acceleration magnitude
        magnitude = np.sqrt(x**2 + y**2 + z**2)
        
        # Stage 1: Peak Detection
        if magnitude > threshold:
            # Stage 2: Validation - check next few samples
            if i + 3 < len(accel_data):
                sustained = any(
                    np.sqrt(accel_data[j][0]**2 + accel_data[j][1]**2 + accel_data[j][2]**2) > threshold * 0.7
                    for j in range(i+1, min(i+4, len(accel_data)))
                )
                
                if sustained:
                    crash_detected = True
                    # Stage 3: Classification
                    if magnitude > 4.0:
                        severity = "severe"
                    elif magnitude > 3.0:
                        severity = "moderate"
                    else:
                        severity = "minor"
                    break
    
    return crash_detected, severity

# Simulate accelerometer data (normal driving vs crash scenario)
print("=== Normal Driving ===")
normal_data = np.random.normal(0, 0.5, (100, 3))  # Low variance, centered at 0G
crash, sev = detect_crash(normal_data)
print(f"Crash Detected: {crash}, Severity: {sev}")

print("\n=== Crash Scenario ===")
crash_data = np.random.normal(0, 0.3, (50, 3))
# Insert crash event at index 25
crash_data[25] = [3.5, 2.8, 4.2]  # High G-force spike
crash_data[26] = [2.1, 1.8, 2.5]  # Sustained impact
crash_data[27] = [1.5, 1.2, 1.8]
crash, sev = detect_crash(crash_data)
print(f"Crash Detected: {crash}, Severity: {sev}")

if crash:
    print("🚨 EMERGENCY: Crash detected! Notifying emergency contacts...")

## Part 4: Model Optimization & Conversion
Convert models to TensorFlow Lite (.tflite) for Android and Core ML (.mlmodel) for iOS to enable on-device inference.

In [ ]:
import tensorflow as tf

# Example: Convert a TensorFlow model to TensorFlow Lite
def convert_to_tflite(model, output_path='model.tflite'):
    """
    Convert TensorFlow/Keras model to TFLite format for Android deployment.
    Includes quantization for reduced size and faster inference.
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Optional: Apply quantization for better performance
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    
    tflite_model = converter.convert()
    
    # Save the model
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"✓ TFLite model saved to {output_path}")
    print(f"  Model size: {len(tflite_model) / 1024:.2f} KB")
    return output_path

# Example: Load MobileNetV2 for distraction classification
print("=== Loading MobileNetV2 for Distraction Classification ===")
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Add custom classification head for distraction types
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dense(128, activation='relu')(x)
output = tf.keras.layers.Dense(4, activation='softmax')(x)  # 4 classes: normal, phone, looking_away, drowsy

model = tf.keras.Model(inputs=base_model.input, outputs=output)
print(f"✓ MobileNetV2 model loaded with {model.count_params():,} parameters")

# Convert to TFLite
print("\n=== Converting to TFLite ===")
tflite_path = convert_to_tflite(model, 'distraction_model.tflite')

print("\n=== Core ML Conversion (iOS) ===")
print("For iOS deployment, use coremltools:")
print("  import coremltools as ct")
print("  coreml_model = ct.convert(model, source='tensorflow')")
print("  coreml_model.save('distraction_model.mlmodel')")

## Part 5: Performance Benchmarking
Measure inference time, FPS, CPU/GPU usage, and battery impact on various devices to ensure real-time performance.

In [ ]:
import time

def benchmark_inference(model, input_shape=(1, 224, 224, 3), num_runs=100):
    """
    Benchmark ML model inference performance.
    
    Returns:
        avg_inference_time (ms), fps, cpu_usage_estimate
    """
    # Generate random input
    dummy_input = np.random.rand(*input_shape).astype(np.float32)
    
    # Warm-up runs
    for _ in range(10):
        _ = model.predict(dummy_input, verbose=0)
    
    # Benchmark runs
    start = time.time()
    for _ in range(num_runs):
        _ = model.predict(dummy_input, verbose=0)
    end = time.time()
    
    total_time = (end - start) * 1000  # Convert to ms
    avg_inference_time = total_time / num_runs
    fps = 1000 / avg_inference_time if avg_inference_time > 0 else 0
    
    return avg_inference_time, fps

print("=== Benchmarking MobileNetV2 Inference ===")
avg_time, fps = benchmark_inference(model, num_runs=50)

print(f"\nResults:")
print(f"  Avg Inference Time: {avg_time:.2f} ms")
print(f"  Estimated FPS: {fps:.1f}")
print(f"\nTarget Performance:")
print(f"  ✓ Target FPS: 30+ (Current: {fps:.1f})")
print(f"  ✓ Target Latency: <50ms (Current: {avg_time:.2f}ms)")

if fps >= 30 and avg_time < 50:
    print("\n✅ PASS: Model meets real-time performance requirements!")
else:
    print("\n⚠️ OPTIMIZATION NEEDED: Consider model quantization or frame skipping")

# Device compatibility matrix (simulated)
print("\n=== Device Compatibility Matrix ===")
devices = [
    {"name": "iPhone 15 Pro", "fps": 45, "latency": 22, "battery_drain": "8%/hr"},
    {"name": "Samsung S23", "fps": 42, "latency": 24, "battery_drain": "9%/hr"},
    {"name": "iPhone 12", "fps": 32, "latency": 31, "battery_drain": "12%/hr"},
    {"name": "Pixel 6", "fps": 35, "latency": 29, "battery_drain": "11%/hr"},
    {"name": "Budget Android", "fps": 18, "latency": 55, "battery_drain": "18%/hr"}
]

for device in devices:
    status = "✓" if device["fps"] >= 25 else "⚠️"
    print(f"{status} {device['name']}: {device['fps']} FPS, {device['latency']}ms, {device['battery_drain']}")

## Summary & Next Steps

This POC demonstrates:
✅ MediaPipe FaceMesh for face landmark detection  
✅ Eye Aspect Ratio (EAR) for drowsiness detection  
✅ Accelerometer-based crash detection algorithm  
✅ TensorFlow Lite model conversion for mobile deployment  
✅ Performance benchmarking framework  

### For May 2026 Implementation:
1. **Data Collection**: Gather labeled dataset for distraction classification (State Farm dataset)
2. **Model Training**: Fine-tune MobileNetV2 on distraction classes
3. **Real-time Pipeline**: Integrate camera → ML → alerts in mobile app
4. **Battery Optimization**: Implement adaptive frame rate (30 FPS → 15 FPS on low battery)
5. **Testing**: Validate on 10+ devices across iOS and Android
6. **Deployment**: Package TFLite and Core ML models for production

### Performance Targets:
- Face Detection: 30+ FPS
- Distraction Accuracy: 85-90%+
- Latency: <50ms
- Battery Drain: <15%/hour